In [ ]:
import os

print(os.path.exists("../data/raw_forensic_atomic.csv"))

In [ ]:
import pandas as pd

# Upload the original dataset
df = pd.read_csv("../data/forensic_atomic_final_v2.csv", engine="python")

# Create a copy
df_clone = df.copy()

print("Original dataset:", df.shape)
print("Clone dataset:", df_clone.shape)

In [ ]:
import re
import json

# Allowlist approach: only permit characters from Latin Unicode blocks.
# This covers standard ASCII, accented European letters (é, à, ü, ñ, ø, etc.),
# and common punctuation/currency symbols.
# Any character outside these blocks (CJK, Arabic, Cyrillic, etc.) will cause the row to be rejected.

_ALLOWED = re.compile(
    r'^['
    r'\x09\x0A\x0D\x20-\x7E'   # Printable ASCII + tab/newline
    r'\u00A0-\u024F'             # Latin-1 Supplement + Latin Extended A + B
    r'\u1E00-\u1EFF'             # Latin Extended Additional (precomposed diacritics)
    r'\u2000-\u206F'             # General punctuation
    r'\u20A0-\u20CF'             # Currency symbols
    r']*$',
    re.UNICODE
)

def _is_latin_safe(text: str) -> bool:
    """Returns True if the text contains only allowed Latin/ASCII characters."""
    return bool(_ALLOWED.match(str(text)))

def row_is_latin_safe(row) -> bool:
    """
    Checks every column in the row, including the contents of JSON list fields
    (xIntent, xEffect, etc.), for non-Latin characters.
    Returns False if even a single disallowed character is found.
    """
    for col in row.index:
        val = row[col]
        if pd.isna(val):
            continue

        val_str = str(val)

        # Try to parse as a JSON list (handles columns like xIntent, xEffect, etc.)
        try:
            parsed = json.loads(val_str)
            if isinstance(parsed, list):
                for item in parsed:
                    if not _is_latin_safe(str(item)):
                        return False
                continue
        except Exception:
            pass

        # Check the raw string value
        if not _is_latin_safe(val_str):
            return False

    return True


# Apply the filter
before = len(df_clone)
df_clean = df_clone[df_clone.apply(row_is_latin_safe, axis=1)].reset_index(drop=True)
after = len(df_clean)

print(f"Rows removed (non-Latin characters): {before - after} ({before} -> {after})")
print(f"Percentage removed: {(before - after) / before * 100:.2f}%")

# Reassign df_clone so all subsequent cells use the filtered dataset
df_clone = df_clean

In [ ]:
# MAPPING: crime_subcategory -> subcategory, done by viewing all the sub-category
SUBCATEGORY_MAP = {
    # Violent Crimes
    "Assault":                    "Assault",
    "Homicide":                   "Homicide",
    "Kidnapping":                 "Kidnapping",
    "Robbery":                    "Robbery",
    "Sexual Offense":             "Sexual Offense",
    "Sexual Offenses":            "Sexual Offense",      # plurals
    "Stalking":                   "Assault",             # Violent Crimes -> Assault
    "Extortion":                  "Robbery",             # Violent Crimes -> Robbery
    "Torture":                    "Assault",             # Violent Crimes -> Assault
    "Threatening Communications": "Assault",             # Violent Crimes -> Assault
    "Witness Intimidation":       "Obstruction of Justice",  # Public Order

    # Property Crimes
    "Theft":                      "Theft",
    "Burglary":                   "Burglary",
    "Arson":                      "Arson",
    "Vandalism":                  "Vandalism",
    "Trespassing":                "Trespassing",

    # Financial Crimes
    "Fraud":                      "Fraud",
    "Insurance Fraud":            "Fraud",               # sottotipo -> Fraud
    "Insider Trading":            "Fraud",               # Financial -> Fraud
    "Counterfeiting":             "Fraud",               # Financial -> Fraud
    "Money Laundering":           "Money Laundering",
    "Embezzlement":               "Embezzlement",
    "Bribery":                    "Bribery",
    "Tax Evasion":                "Tax Evasion",

    # Cyber Crimes
    "Hacking":                    "Hacking",
    "Phishing":                   "Phishing",
    "Identity Theft":             "Identity Theft",
    "Online Harassment":          "Online Harassment",
    "Planning a Cyber Attack":    "Hacking",             # Cyber -> Hacking

    # Organized Crimes
    "Drug Trafficking":           "Drug Trafficking",
    "Human Trafficking":          "Human Trafficking",
    "Racketeering":               "Racketeering",
    "Weapon Smuggling":           "Weapon Smuggling",
    "Weapon Offense":             "Weapon Smuggling",    # Organized / Weapon Smuggling
    "Weapon Possession":          "Weapon Smuggling",    # Organized / Weapon Smuggling
    "Gang Activity":              "Racketeering",        # Organized / Racketeering
    "Terrorism":                  "Racketeering",        # Organized / Racketeering
    "Animal Fighting":            "Racketeering",
    "Dog Fighting":               "Racketeering",
    "Animal Trafficking":         "Human Trafficking",   # trafficking -> Organized
    "Wildlife Trafficking":       "Human Trafficking",
    "Animal Cruelty":             "Racketeering",        # associated to organized crimes
    "Cruelty to Animals":         "Racketeering",
    # a long description
    "Animal Cruelty (implied within trafficking or illegal breeding operations, categorized here as a component of Organized Crime for profit-driven animal exploitation)": "Racketeering",
    "Organized Crimes":           "Racketeering",        # generic label -> canonic

    # Public Order/Justice
    "Perjury":                    "Perjury",
    "Obstruction of Justice":     "Obstruction of Justice",
    "Disorderly Conduct":         "Disorderly Conduct",
}

# MAPPING: subcategory -> category
SUBCATEGORY_TO_CATEGORY = {
    # Violent Crimes
    "Assault":                   "Violent Crimes",
    "Homicide":                  "Violent Crimes",
    "Kidnapping":                "Violent Crimes",
    "Robbery":                   "Violent Crimes",
    "Sexual Offense":            "Violent Crimes",
    # Property Crimes
    "Theft":                     "Property Crimes",
    "Burglary":                  "Property Crimes",
    "Arson":                     "Property Crimes",
    "Vandalism":                 "Property Crimes",
    "Trespassing":               "Property Crimes",
    # Financial Crimes
    "Fraud":                     "Financial Crimes",
    "Money Laundering":          "Financial Crimes",
    "Embezzlement":              "Financial Crimes",
    "Bribery":                   "Financial Crimes",
    "Tax Evasion":               "Financial Crimes",
    # Cyber Crimes
    "Hacking":                   "Cyber Crimes",
    "Phishing":                  "Cyber Crimes",
    "Identity Theft":            "Cyber Crimes",
    "Online Harassment":         "Cyber Crimes",
    # Organized Crimes
    "Drug Trafficking":          "Organized Crimes",
    "Human Trafficking":         "Organized Crimes",
    "Racketeering":              "Organized Crimes",
    "Weapon Smuggling":          "Organized Crimes",
    # Public Order/Justice
    "Perjury":                   "Public Order/Justice",
    "Obstruction of Justice":    "Public Order/Justice",
    "Disorderly Conduct":        "Public Order/Justice",
}

print("Mappings defined.")
print(f"Subcategory map entries : {len(SUBCATEGORY_MAP)}")
print(f"Category map entries    : {len(SUBCATEGORY_TO_CATEGORY)}")

In [ ]:
# Mapping

# Normalize crime_subcategory
df_clone["crime_subcategory"] = (
    df_clone["crime_subcategory"]
    .str.strip()
    .map(SUBCATEGORY_MAP)
)

unmapped_sub = df_clone["crime_subcategory"].isna().sum()
print(f"Subcategory non mappate (NaN): {unmapped_sub}")

# Derive crime_category from the normalized subcategory
df_clone["crime_category"] = (
    df_clone["crime_subcategory"]
    .map(SUBCATEGORY_TO_CATEGORY)
)

unmapped_cat = df_clone["crime_category"].isna().sum()
print(f"Category not mapped (NaN): {unmapped_cat}")

In [ ]:
# REPORT CLEANING
original_rows = df.shape[0]
final_rows    = df_clone.shape[0]
removed_rows  = original_rows - final_rows

print("-" * 50)
print("DATASET CLEANING REPORT")
print("-" * 50)
print(f"Original Rows   : {original_rows}")
print(f"Final Rows      : {final_rows}")
print(f"Removed Rows    : {removed_rows}")
print(f"% removed       : {removed_rows / original_rows * 100:.2f}%")

In [ ]:
# CRIME CATEGORY DISTRIBUTION
print("\nCRIME CATEGORY — Distribution")
print("=" * 50)
cat_dist = df_clone["crime_category"].value_counts()
for cat, count in cat_dist.items():
    pct = count / final_rows * 100
    print(f"  {cat:<35} {count:>5} rows  ({pct:.1f}%)")
print(f"  {'TOTAL':<35} {final_rows:>5} rows")

In [ ]:
# CRIME SUBCATEGORY DISTRIBUTION
print("\nCRIME SUBCATEGORY — Distribuzione")
print("=" * 50)
sub_dist = df_clone["crime_subcategory"].value_counts()
for sub, count in sub_dist.items():
    pct = count / final_rows * 100
    print(f"  {sub:<40} {count:>5} rows  ({pct:.1f}%)")
print(f"  {'TOTAL':<40} {final_rows:>5} rows")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Category
cat_dist.plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Crime Category (normalized)")
axes[0].set_xlabel("Category")
axes[0].set_ylabel("Number of events")
axes[0].tick_params(axis='x', rotation=45)

# Subcategory (top 15)
sub_dist.head(15).plot(kind="bar", ax=axes[1], color="darkorange")
axes[1].set_title("Crime Subcategory — Top 15 (normalized)")
axes[1].set_xlabel("Subcategory")
axes[1].set_ylabel("Number of events")
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# SAVING FILTERED ATOMIC
output_path = "../data/pre-judge_forensic_atomic.csv"
df_clone.to_csv(output_path, index=False)
print(f"Saved in: {output_path}")
print(f"Shape: {df_clone.shape}")
display(df_clone.head())